# Impulse — A2D2 Event Verification (VLM-as-judge)

Companion to `a2d2_ingestion.ipynb`, `a2d2_object_detection.ipynb`, and `a2d2_analysis.ipynb`.

The 7 mined events are pure **threshold rules** over bus + perception channels, so they fire on
noise and on scenes that aren't actually the event of interest (a misranged detection, a parked /
irrelevant object, a 1-sample kinematic blip with no cause in the scene). This notebook adds an
**automatic relevance check**: for every detected event it samples a few camera frames inside the
event window, sends them to a **Databricks-hosted multimodal model** with an **event-specific
rubric** (plus the within-event telemetry stats as grounding), and stores a structured verdict
`{is_relevant, confidence, reason}` in `{prefix}_event_verification_fact`. The Event Explorer app
LEFT-JOINs that table to badge / filter events.

## Prerequisite
Run ingestion (with `download_images=true`), object detection, and analysis first. This notebook
reads `{prefix}_event_instance_fact`, `{prefix}_event_dimension`, `{prefix}_camera_frames`, and
`{prefix}_stats_aggregator_fact`, and calls the serving endpoint named by `verify_model`.

## License
A2D2 © Audi AG, **CC BY-ND 4.0** (https://creativecommons.org/licenses/by-nd/4.0/, source
https://www.a2d2.audi/). This notebook reads only your own ingested tables/images and stores only
a verdict row per event — it redistributes nothing. Commit it with outputs cleared.

In [ ]:
%pip install pillow -q
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
dbutils.widgets.text("verify_model", "databricks-claude-opus-4-8", "Multimodal serving endpoint")
dbutils.widgets.text("frames_per_event", "3", "Frames sent to the judge per event")
dbutils.widgets.text("verify_window_s", "6", "Max seconds from event start to sample frames from")
dbutils.widgets.text("max_image_width", "512", "Downscale frames to this width (px) before sending")
dbutils.widgets.text("concurrency", "2", "Parallel model calls (keep low: FM endpoints rate-limit)")
dbutils.widgets.text("verify_limit", "0", "Verify at most N events (0 = all; for smoke tests)")

In [ ]:
import sys, os, time

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
VERIFY_MODEL = dbutils.widgets.get("verify_model")
FRAMES_PER_EVENT = int(dbutils.widgets.get("frames_per_event") or "3")
VERIFY_WINDOW_S = float(dbutils.widgets.get("verify_window_s") or "6")
MAX_IMAGE_WIDTH = int(dbutils.widgets.get("max_image_width") or "512")
CONCURRENCY = int(dbutils.widgets.get("concurrency") or "8")
VERIFY_LIMIT = int(dbutils.widgets.get("verify_limit") or "0")

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME and VERIFY_MODEL):
    raise ValueError("Please set catalog, schema, table_prefix, volume and verify_model widgets above.")

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
VERIFY_TABLE = f"{pfx}_event_verification_fact"
for t in ("event_instance_fact", "event_dimension", "camera_frames"):
    if not spark.catalog.tableExists(f"{pfx}_{t}"):
        raise RuntimeError(f"{pfx}_{t} not found. Run ingestion + analysis first.")

# Workspace host + token for the serving-endpoint REST call (OpenAI-compatible chat API).
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()
print(f"Verifying events in {pfx}_* with model '{VERIFY_MODEL}' -> {VERIFY_TABLE}")

## 1. Load events, telemetry context, and frame index

One row per detected event instance (joined to its name/type), the per-event channel statistics
already computed by the analysis notebook (used to ground the prompt), and the camera-frame index
(container + timestamp + PNG path) so we can pick representative frames per event.

In [ ]:
import pyspark.sql.functions as F

events_pdf = (spark.read.table(f"{pfx}_event_instance_fact")
              .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name", "event_type"),
                    "event_id")
              .select("container_id", "event_instance_id", "event_id", "event_name", "event_type",
                      "start_ts", "end_ts")
              .orderBy("event_name", "container_id", "start_ts")
              .toPandas())

if VERIFY_LIMIT and VERIFY_LIMIT > 0:
    events_pdf = events_pdf.head(VERIFY_LIMIT).copy()

frames_pdf = (spark.read.table(f"{pfx}_camera_frames")
              .select("container_id", "cam_tstamp_us", "png_path")
              .orderBy("cam_tstamp_us").toPandas())

# Per-(container, instance) telemetry stats -> {channel: {label: value}}; used as prompt grounding.
stats_pdf = (spark.read.table(f"{pfx}_stats_aggregator_fact")
             .select("container_id", "event_instance_id", "channel_name", "aggregation_label", "statistic_value")
             .toPandas()) if spark.catalog.tableExists(f"{pfx}_stats_aggregator_fact") else None

STATS = {}
if stats_pdf is not None:
    for _, r in stats_pdf.iterrows():
        key = (int(r["container_id"]), int(r["event_instance_id"]))
        ch = STATS.setdefault(key, {}).setdefault(r["channel_name"], {})
        v = r["statistic_value"]
        ch[r["aggregation_label"]] = (None if v is None else float(v))

print(f"{len(events_pdf)} event instance(s); {len(frames_pdf)} frames; "
      f"{len(STATS)} instance(s) with telemetry stats.")

## 2. Frame selection + JPEG encoding

For each event take up to `frames_per_event` **evenly-spaced** frames within
`[start_ts, min(end_ts, start_ts + verify_window_s)]` (chronological), downscale to
`max_image_width`, and base64-encode as JPEG data URIs. Same window logic as the clip exporter in
`a2d2_analysis.ipynb`, but capped to a handful of frames.

In [ ]:
import io, base64
import numpy as np
from PIL import Image

def select_frame_paths(container_id, start_ts, end_ts, n, window_s):
    """Up to n evenly-spaced frame paths of THIS container within the (capped) event window."""
    end = min(int(end_ts), int(start_ts) + int(window_s * 1e6))
    sub = frames_pdf[(frames_pdf.container_id == container_id)
                     & (frames_pdf.cam_tstamp_us >= start_ts)
                     & (frames_pdf.cam_tstamp_us <= end)].sort_values("cam_tstamp_us")
    if len(sub) == 0:
        return []
    if len(sub) <= n:
        return sub.png_path.tolist()
    idx = np.linspace(0, len(sub) - 1, n).round().astype(int)
    return sub.iloc[idx].png_path.tolist()

def encode_jpeg_b64(path, max_width):
    im = Image.open(path).convert("RGB")
    w, h = im.size
    if w > max_width:
        im = im.resize((max_width, max(1, int(round(h * max_width / w)))))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode("ascii")

## 3. Event-specific rubrics + prompt

Each event type gets a tailored rubric describing what the scene/telemetry should show and when to
call it a false positive. Object-presence events ask the judge to confirm the VRU is actually
present and in the relevant position; kinematic events ask it to confirm a plausible cause across
the frame sequence, leaning on the telemetry summary.

In [ ]:
SYSTEM = (
    "You are a meticulous ADAS validation reviewer. You are shown forward-facing camera frames "
    "(in chronological order) sampled from a vehicle during a CANDIDATE driving event that was "
    "flagged by threshold rules over vehicle telemetry. Your job is to decide whether the frames "
    "genuinely show the event of interest, or whether it is a FALSE POSITIVE. Be strict and "
    "conservative: if the scene does not support the event, mark it not relevant. "
    'Respond with ONLY a compact JSON object and nothing else: '
    '{"is_relevant": true|false, "confidence": <0.0-1.0>, "reason": "<=20 words"}.'
)

RUBRICS = {
    "emergency_braking":
        "Event: hard / emergency braking. A genuine case shows a plausible cause ahead — a close "
        "lead vehicle, brake lights, a pedestrian or obstacle, a red light / junction, or dense or "
        "slowing traffic. FALSE POSITIVE if the road ahead is clearly open with no hazard and the "
        "deceleration looks like a minor, routine speed adjustment.",
    "sharp_cornering":
        "Event: sharp / aggressive cornering. A genuine case is a tight bend, a sharp turn at a "
        "junction, or a swerve (steering visibly changes across the frames). FALSE POSITIVE if the "
        "vehicle is going essentially straight or following a gentle, ordinary curve.",
    "evasive_maneuver":
        "Event: evasive brake-then-swerve to avoid a hazard. A genuine case has an obstacle, "
        "vehicle, or vulnerable road user prompting avoidance. FALSE POSITIVE if no hazard is "
        "present and the motion looks like normal driving.",
    "pedestrian_near_miss":
        "Event: pedestrian near-miss. A genuine case has a pedestrian actually present and close to "
        "the vehicle's trajectory. FALSE POSITIVE if no pedestrian is visible, the person is far "
        "away or safely behind a barrier, or it is a misdetection (sign, poster, reflection).",
    "cyclist_close_encounter":
        "Event: close encounter with a cyclist / motorcyclist. A genuine case has a cyclist (or "
        "motorcyclist) actually present within a few metres of the vehicle. FALSE POSITIVE if none "
        "is visible or it is a misdetection.",
    "vru_brake_reaction":
        "Event: braking in reaction to a vulnerable road user. A genuine case shows a pedestrian or "
        "cyclist present AND a scene consistent with braking for them. FALSE POSITIVE if no "
        "vulnerable road user is visible.",
    "pedestrian_in_path":
        "Event: pedestrian directly in the vehicle's path. A genuine case has a pedestrian ahead, "
        "in or entering the vehicle's lane / forward path, at close range. FALSE POSITIVE if the "
        "only people are far to the side / on the sidewalk, or it is a misdetection.",
}
DEFAULT_RUBRIC = ("Decide whether the frames genuinely show the named event, or whether the "
                  "threshold rule fired without a real corresponding situation in the scene.")

def stats_text(container_id, instance_id):
    d = STATS.get((int(container_id), int(instance_id)))
    if not d:
        return "(no telemetry summary available)"
    parts = []
    for ch, labels in sorted(d.items()):
        bits = []
        for lab in ("min", "mean", "max"):
            if lab in labels and labels[lab] is not None:
                bits.append(f"{lab}={labels[lab]:.1f}")
        if bits:
            parts.append(f"{ch}: " + ", ".join(bits))
    return "; ".join(parts) if parts else "(no telemetry summary available)"

def build_user_text(event_name, container_id, instance_id, n_frames):
    rubric = RUBRICS.get(event_name, DEFAULT_RUBRIC)
    return (f"Candidate event: '{event_name}'. {rubric}\n\n"
            f"Telemetry during the event window: {stats_text(container_id, instance_id)}.\n\n"
            f"You are given {n_frames} frame(s) in chronological order. "
            f"Does this genuinely show '{event_name}'?")

## 4. Call the multimodal judge (parallel REST)

Each event becomes one OpenAI-compatible chat request to the `verify_model` serving endpoint: the
rubric text plus the frames as `image_url` data URIs. We parse the JSON verdict defensively. Calls
run in a small thread pool — there are only a few dozen events.

In [ ]:
import json, re, time, random, requests
from concurrent.futures import ThreadPoolExecutor

INVOKE_URL = f"{HOST}/serving-endpoints/{VERIFY_MODEL}/invocations"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def _invoke(payload, max_retries=7):
    """POST to the serving endpoint, retrying on 429/5xx with exponential backoff + jitter
    (honouring Retry-After). Returns the assistant message text, or raises with the body."""
    delay = 2.0
    for attempt in range(max_retries + 1):
        resp = requests.post(INVOKE_URL, headers=HEADERS, json=payload, timeout=180)
        if resp.status_code == 200:
            return resp.json()["choices"][0]["message"]["content"]
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
            ra = resp.headers.get("Retry-After")
            wait = float(ra) if ra and ra.replace(".", "", 1).isdigit() else delay
            time.sleep(wait + random.uniform(0, 1.0))
            delay = min(delay * 2, 60.0)
            continue
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
    raise RuntimeError("exhausted retries")

def _parse_verdict(text):
    """Pull the first JSON object out of the model's reply, defensively."""
    if not text:
        raise ValueError("empty reply")
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t).rstrip("`").strip()
    i, j = t.find("{"), t.rfind("}")
    if i == -1 or j == -1:
        raise ValueError(f"no JSON in reply: {text[:120]!r}")
    obj = json.loads(t[i:j + 1])
    rel = obj.get("is_relevant")
    if isinstance(rel, str):
        rel = rel.strip().lower() in ("true", "yes", "1", "relevant")
    conf = obj.get("confidence")
    conf = float(conf) if conf is not None else None
    if conf is not None:
        conf = max(0.0, min(1.0, conf))
    reason = str(obj.get("reason", ""))[:240]
    return bool(rel), conf, reason

def verify_one(row):
    cid = int(row["container_id"]); eiid = int(row["event_instance_id"]); eid = int(row["event_id"])
    name = row["event_name"]
    base = {"container_id": cid, "event_instance_id": eiid, "event_id": eid,
            "is_relevant": None, "confidence": None, "reason": None,
            "frames_used": 0, "model": VERIFY_MODEL}
    try:
        paths = select_frame_paths(cid, row["start_ts"], row["end_ts"], FRAMES_PER_EVENT, VERIFY_WINDOW_S)
        imgs = []
        for p in paths:
            try:
                imgs.append(encode_jpeg_b64(p, MAX_IMAGE_WIDTH))
            except Exception:
                continue
        if not imgs:
            base["reason"] = "no frames available for this event window"
            return base
        content = [{"type": "text", "text": build_user_text(name, cid, eiid, len(imgs))}]
        content += [{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b}"}} for b in imgs]
        # NB: claude-opus-4-8 rejects the `temperature` parameter, so we omit it.
        payload = {"messages": [{"role": "system", "content": SYSTEM},
                                {"role": "user", "content": content}],
                   "max_tokens": 300}
        reply = _invoke(payload)
        rel, conf, reason = _parse_verdict(reply)
        base.update(is_relevant=rel, confidence=conf, reason=reason, frames_used=len(imgs))
    except Exception as e:  # noqa: BLE001
        base["reason"] = f"verify_error: {str(e)[:200]}"
    return base

rows = [r for _, r in events_pdf.iterrows()]
t0 = time.time()
with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
    results = list(ex.map(verify_one, rows))
ok = sum(1 for r in results if r["is_relevant"] is not None)
print(f"Verified {ok}/{len(results)} events in {time.time() - t0:.0f}s "
      f"({len(results) - ok} errored/skipped).")

## 5. Persist verdicts

One row per event instance in `{prefix}_event_verification_fact`. Idempotent — the table is fully
rebuilt each run.

In [ ]:
from pyspark.sql.types import (StructType, StructField, IntegerType, BooleanType,
                                       DoubleType, StringType, LongType)

VERIFY_SCHEMA = StructType([
    StructField("container_id", IntegerType(), False),
    StructField("event_instance_id", LongType(), False),  # full 32-bit CRC32 -> needs long
    StructField("event_id", IntegerType(), False),
    StructField("is_relevant", BooleanType(), True),
    StructField("confidence", DoubleType(), True),
    StructField("reason", StringType(), True),
    StructField("frames_used", IntegerType(), True),
    StructField("model", StringType(), True),
    StructField("verified_ts", LongType(), True),
])

now_us = int(time.time() * 1e6)
out = [(r["container_id"], r["event_instance_id"], r["event_id"], r["is_relevant"],
        r["confidence"], r["reason"], r["frames_used"], r["model"], now_us) for r in results]

if out:
    (spark.createDataFrame(out, schema=VERIFY_SCHEMA)
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(VERIFY_TABLE))
    print(f"Wrote {len(out)} verdict row(s) to {VERIFY_TABLE}.")
else:
    print("No events to verify; table not written.")

## 6. Summary

In [ ]:
if out:
    v = spark.read.table(VERIFY_TABLE).join(
        spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id")
    print("Relevant vs flagged per event type:")
    display(v.groupBy("event_name").agg(
        F.count("*").alias("events"),
        F.sum(F.col("is_relevant").cast("int")).alias("relevant"),
        F.sum((~F.col("is_relevant")).cast("int")).alias("flagged"),
        F.sum(F.col("is_relevant").isNull().cast("int")).alias("unknown"),
        F.round(F.avg("confidence"), 2).alias("avg_conf"),
    ).orderBy("event_name"))
    print("Flagged / low-confidence sample (eyeball these against their clips):")
    display(v.where("is_relevant = false OR confidence < 0.6 OR is_relevant IS NULL")
            .select("event_name", "container_id", "event_instance_id", "is_relevant",
                    "confidence", "reason")
            .orderBy("confidence").limit(25))